In [1]:
# Cell 0 | 导入包、固定随机种子并设置实验路径

import copy
import hashlib
import json
import os
import random
import sys

from datetime import datetime
from pathlib import Path
from typing import Union

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)
from torch_geometric.data import Batch
from torch_geometric.nn import (
    NNConv,
    global_mean_pool,
)


def find_project_root(
    start_path: Union[str, Path] = ".",
) -> Path:
    """向上查找同时包含 data 和 notebooks 的项目根目录。"""
    current_path = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认 notebook 位于 "
        r"D:\jupyter\IL_ML_Project"
        " 项目文件夹内。"
    )


def seed_everything(seed: int) -> None:
    """固定随机种子，尽量保证结果可复现。"""
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


ROOT = find_project_root(Path.cwd())

EXPERIMENT_ROOT = (
    ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

GNN_DIR = (
    EXPERIMENT_ROOT
    / "features"
    / "mpnn_graph"
)

REPRESENTATION_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
)

PRETRAIN_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "mpnn_pretraining"
)

MODEL_DIR = (
    EXPERIMENT_ROOT
    / "models"
    / "mpnn_feature_extractors"
)

for folder in [
    REPRESENTATION_DIR,
    PRETRAIN_DIR,
    MODEL_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# 04 notebook 生成的输入文件
CATION_GRAPHS_PATH = (
    GNN_DIR
    / "cation_graphs.pt"
)

ANION_GRAPHS_PATH = (
    GNN_DIR
    / "anion_graphs.pt"
)

GRAPH_VALID_PATH = (
    GNN_DIR
    / "graph_valid.npz"
)

GRAPH_MANIFEST_PATH = (
    GNN_DIR
    / "graph_manifest.csv"
)

DF_OK_PATH = (
    REPRESENTATION_DIR
    / "df_ok_feature_representation.csv"
)

FEATURE_REPRESENTATION_MANIFEST_PATH = (
    REPRESENTATION_DIR
    / "feature_representation_manifest.json"
)

# 本 notebook 的输出文件
FEATURE_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta.npy"
)

FEATURE_ROW_MANIFEST_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta_row_manifest.csv"
)

FEATURE_ALIGNMENT_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta_alignment.npz"
)

INTERNAL_SPLIT_PATH = (
    PRETRAIN_DIR
    / "mpnn_pretraining_internal_split_indices.npz"
)

TRAINING_HISTORY_PATH = (
    PRETRAIN_DIR
    / "mpnn_pretraining_history_eta.csv"
)

PRETRAIN_METRICS_PATH = (
    PRETRAIN_DIR
    / "mpnn_pretraining_metrics_eta.csv"
)

VAL_PREDICTIONS_PATH = (
    PRETRAIN_DIR
    / "mpnn_pretraining_val_predictions_eta.csv"
)

MODEL_PATH = (
    MODEL_DIR
    / "mpnn_feature_extractor_viscosity.pt"
)

TP_SCALER_PATH = (
    MODEL_DIR
    / "tp_scaler_viscosity.joblib"
)

TARGET_SCALER_PATH = (
    MODEL_DIR
    / "target_scaler_viscosity.joblib"
)

MODEL_METADATA_PATH = (
    MODEL_DIR
    / "mpnn_feature_extractor_viscosity_metadata.json"
)

# 防止误写回大论文原目录
experiment_text = str(
    EXPERIMENT_ROOT.resolve()
).lower()

if (
    "small_paper_data" not in experiment_text
    or "viscosity_10point_holdout" not in experiment_text
):
    raise RuntimeError(
        "10点留出实验目录异常，已停止运行，"
        "防止覆盖大论文原始文件。"
    )

required_input_files = [
    CATION_GRAPHS_PATH,
    ANION_GRAPHS_PATH,
    GRAPH_VALID_PATH,
    GRAPH_MANIFEST_PATH,
    DF_OK_PATH,
    FEATURE_REPRESENTATION_MANIFEST_PATH,
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        "缺少 04 notebook 生成的输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_input_files
        )
    )

SEED = 42
seed_everything(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

PIN_MEMORY = (
    device.type == "cuda"
)

print("python                 =", sys.executable)
print("device                 =", device)
print("random seed            =", SEED)
print("ROOT                   =", ROOT)
print("EXPERIMENT_ROOT        =", EXPERIMENT_ROOT)
print("GNN_DIR                =", GNN_DIR)
print("REPRESENTATION_DIR     =", REPRESENTATION_DIR)
print("PRETRAIN_DIR           =", PRETRAIN_DIR)
print("MODEL_DIR              =", MODEL_DIR)

python                 = D:\conda\envs\rdkit-env\python.exe
device                 = cpu
random seed            = 42
ROOT                   = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
GNN_DIR                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph
REPRESENTATION_DIR     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare
PRETRAIN_DIR           = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining
MODEL_DIR              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\models\mpnn_feature_extractors


In [2]:
# Cell 1 | 读取分子图并与黏度数据严格对齐


def load_torch_object(path: Path):
    """兼容不同 PyTorch 版本读取对象。"""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


with open(
    FEATURE_REPRESENTATION_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_representation_manifest = json.load(file)

if (
    feature_representation_manifest.get(
        "target_property"
    )
    != "viscosity"
):
    raise ValueError(
        "04 notebook 的 manifest 中 "
        "target_property 不是 viscosity。"
    )

if (
    feature_representation_manifest.get(
        "target_for_modeling"
    )
    != "lg_eta"
):
    raise ValueError(
        "04 notebook 的 manifest 中 "
        "target_for_modeling 不是 lg_eta。"
    )

if (
    feature_representation_manifest.get(
        "representation_method"
    )
    != "MPNN"
):
    raise ValueError(
        "04 notebook 的 manifest 中 "
        "representation_method 不是 MPNN。"
    )

df_manifest = pd.read_csv(
    GRAPH_MANIFEST_PATH,
    encoding="utf-8-sig",
)

df_ok = pd.read_csv(
    DF_OK_PATH,
    encoding="utf-8-sig",
)

df_manifest.columns = (
    df_manifest.columns
    .astype(str)
    .str.strip()
)

df_ok.columns = (
    df_ok.columns
    .astype(str)
    .str.strip()
)

required_manifest_cols = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
    "graph_valid",
    "graph_list_idx",
]

missing_manifest_cols = [
    col
    for col in required_manifest_cols
    if col not in df_manifest.columns
]

if missing_manifest_cols:
    raise ValueError(
        "graph_manifest.csv 缺少必要字段："
        f"{missing_manifest_cols}"
    )

required_df_ok_cols = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

missing_df_ok_cols = [
    col
    for col in required_df_ok_cols
    if col not in df_ok.columns
]

if missing_df_ok_cols:
    raise ValueError(
        "df_ok_feature_representation.csv "
        f"缺少必要字段：{missing_df_ok_cols}"
    )

for col in [
    "source_row_idx",
    "graph_valid",
    "graph_list_idx",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    df_manifest[col] = pd.to_numeric(
        df_manifest[col],
        errors="coerce",
    )

for col in [
    "source_row_idx",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    df_ok[col] = pd.to_numeric(
        df_ok[col],
        errors="coerce",
    )

if df_manifest["source_row_idx"].duplicated().any():
    raise ValueError(
        "graph_manifest.csv 中的 "
        "source_row_idx 存在重复。"
    )

if df_ok["source_row_idx"].duplicated().any():
    raise ValueError(
        "df_ok_feature_representation.csv 中的 "
        "source_row_idx 存在重复。"
    )

manifest_source_idx = (
    df_manifest["source_row_idx"]
    .to_numpy(dtype=int)
)

df_ok_source_idx = (
    df_ok["source_row_idx"]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    manifest_source_idx,
    df_ok_source_idx,
):
    raise ValueError(
        "graph_manifest.csv 与 "
        "df_ok_feature_representation.csv "
        "的 source_row_idx 顺序不一致。"
    )

all_cat_graphs = load_torch_object(
    CATION_GRAPHS_PATH
)

all_ani_graphs = load_torch_object(
    ANION_GRAPHS_PATH
)

valid_manifest = (
    df_manifest[
        df_manifest["graph_valid"].eq(1)
    ]
    .copy()
)

valid_manifest["graph_list_idx"] = (
    valid_manifest["graph_list_idx"]
    .astype(int)
)

valid_manifest = (
    valid_manifest
    .sort_values("graph_list_idx")
    .reset_index(drop=True)
)

expected_graph_list_idx = np.arange(
    len(valid_manifest),
    dtype=int,
)

actual_graph_list_idx = (
    valid_manifest["graph_list_idx"]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    actual_graph_list_idx,
    expected_graph_list_idx,
):
    raise ValueError(
        "graph_manifest.csv 中的 graph_list_idx "
        "与 pt 图列表顺序不一致。"
    )

if not (
    len(all_cat_graphs)
    == len(all_ani_graphs)
    == len(valid_manifest)
):
    raise ValueError(
        "阳离子图、阴离子图与有效图清单数量不一致：\n"
        f"cation={len(all_cat_graphs)}, "
        f"anion={len(all_ani_graphs)}, "
        f"manifest={len(valid_manifest)}"
    )

model_rows = valid_manifest.copy()

for col in [
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    model_rows[col] = pd.to_numeric(
        model_rows[col],
        errors="coerce",
    )

invalid_model_mask = (
    model_rows["T_rep"].isna()
    | model_rows["P_rep"].isna()
    | model_rows["eta"].isna()
    | (model_rows["eta"] <= 0)
    | model_rows["lg_eta"].isna()
)

if invalid_model_mask.any():
    raise ValueError(
        "有效分子图清单中仍存在缺失的 "
        "T、P、eta 或 lg_eta。"
    )

lg_eta_recalculated = np.log10(
    model_rows["eta"].to_numpy(
        dtype=float
    )
)

max_lg_eta_difference = float(
    np.max(
        np.abs(
            model_rows["lg_eta"].to_numpy(
                dtype=float
            )
            - lg_eta_recalculated
        )
    )
)

if max_lg_eta_difference > 1e-8:
    raise ValueError(
        "graph_manifest.csv 中 lg_eta "
        "与 log10(eta) 不一致。\n"
        f"最大绝对差值：{max_lg_eta_difference}"
    )

graph_list_indices = (
    model_rows["graph_list_idx"]
    .to_numpy(dtype=int)
)

cat_graphs = [
    all_cat_graphs[index]
    for index in graph_list_indices
]

ani_graphs = [
    all_ani_graphs[index]
    for index in graph_list_indices
]

env_vars_raw = (
    model_rows[
        [
            "T_rep",
            "P_rep",
        ]
    ]
    .to_numpy(dtype=float)
)

targets_raw = (
    model_rows["lg_eta"]
    .to_numpy(dtype=float)
)

expected_rows = int(
    feature_representation_manifest.get(
        "n_graph_valid_rows",
        -1,
    )
)

if expected_rows != len(model_rows):
    raise ValueError(
        "04 notebook manifest 中的有效图数量 "
        "与当前读取数量不一致。"
    )

print("04 notebook 输出核对通过 ✓")
print("target column         =", "lg_eta")
print("model rows            =", len(model_rows))
print("cation graphs         =", len(cat_graphs))
print("anion graphs          =", len(ani_graphs))
print("unique ILs            =", model_rows["IL_key"].nunique())
print("unique cations        =", model_rows["cation_smiles"].nunique())
print("unique anions         =", model_rows["anion_smiles"].nunique())
print("T range               =", (
    model_rows["T_rep"].min(),
    model_rows["T_rep"].max(),
))
print("P range               =", (
    model_rows["P_rep"].min(),
    model_rows["P_rep"].max(),
))
print("lg_eta range          =", (
    model_rows["lg_eta"].min(),
    model_rows["lg_eta"].max(),
))
print("max |lg_eta-log10η|   =", max_lg_eta_difference)

04 notebook 输出核对通过 ✓
target column         = lg_eta
model rows            = 16216
cation graphs         = 16216
anion graphs          = 16216
unique ILs            = 1295
unique cations        = 583
unique anions         = 202
T range               = (191.2, 573.0)
P range               = (0.02, 9500.0)
lg_eta range          = (-1.0, 12.30102999566398)
max |lg_eta-log10η|   = 1.7763568394002505e-15


In [3]:
# Cell 2 | 划分内部训练集和验证集并建立 DataLoader


class ILDataset(Dataset):
    """离子液体阳离子图、阴离子图、环境条件和目标值数据集。"""

    def __init__(
        self,
        cat,
        ani,
        env,
        target,
    ):
        if not (
            len(cat)
            == len(ani)
            == len(env)
            == len(target)
        ):
            raise ValueError(
                "图、环境变量与目标值数量不一致。"
            )

        self.cat = cat
        self.ani = ani

        self.env = torch.tensor(
            env,
            dtype=torch.float32,
        )

        self.target = torch.tensor(
            target,
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.target)

    def __getitem__(self, index):
        return (
            self.cat[index],
            self.ani[index],
            self.env[index],
            self.target[index],
        )


def custom_collate(batch):
    """将多个 PyG 分子图组合为批次。"""
    cat_list = [
        item[0]
        for item in batch
    ]

    ani_list = [
        item[1]
        for item in batch
    ]

    env_list = [
        item[2]
        for item in batch
    ]

    target_list = [
        item[3]
        for item in batch
    ]

    return (
        Batch.from_data_list(
            cat_list
        ),
        Batch.from_data_list(
            ani_list
        ),
        torch.stack(
            env_list
        ),
        torch.stack(
            target_list
        ),
    )


all_sample_indices = np.arange(
    len(model_rows),
    dtype=int,
)

internal_train_idx, internal_val_idx = train_test_split(
    all_sample_indices,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

# 只使用内部训练部分拟合标准化器
scaler_X = StandardScaler()

scaler_X.fit(
    env_vars_raw[
        internal_train_idx
    ]
)

env_vars_scaled = scaler_X.transform(
    env_vars_raw
).astype(np.float32)

scaler_y = StandardScaler()

scaler_y.fit(
    targets_raw[
        internal_train_idx
    ].reshape(-1, 1)
)

targets_scaled = scaler_y.transform(
    targets_raw.reshape(-1, 1)
).astype(np.float32)

dataset = ILDataset(
    cat=cat_graphs,
    ani=ani_graphs,
    env=env_vars_scaled,
    target=targets_scaled,
)

train_dataset = Subset(
    dataset,
    internal_train_idx.tolist(),
)

val_dataset = Subset(
    dataset,
    internal_val_idx.tolist(),
)

BATCH_SIZE = 128
NUM_WORKERS = 0

train_generator = (
    torch.Generator()
    .manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=train_generator,
)

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

full_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

np.savez_compressed(
    INTERNAL_SPLIT_PATH,

    internal_train_feature_row_idx=(
        internal_train_idx
    ),

    internal_val_feature_row_idx=(
        internal_val_idx
    ),

    internal_train_source_row_idx=(
        model_rows.iloc[
            internal_train_idx
        ]["source_row_idx"]
        .to_numpy(dtype=int)
    ),

    internal_val_source_row_idx=(
        model_rows.iloc[
            internal_val_idx
        ]["source_row_idx"]
        .to_numpy(dtype=int)
    ),

    random_seed=np.array(
        [SEED],
        dtype=int,
    ),
)

print("内部数据划分完成 ✓")
print("all rows              =", len(dataset))
print("internal train rows   =", len(train_dataset))
print("internal val rows     =", len(val_dataset))
print("batch size            =", BATCH_SIZE)
print("scaler_X fitted rows  =", len(internal_train_idx))
print("scaler_y fitted rows  =", len(internal_train_idx))
print("saved:", INTERNAL_SPLIT_PATH)

内部数据划分完成 ✓
all rows              = 16216
internal train rows   = 12972
internal val rows     = 3244
batch size            = 128
scaler_X fitted rows  = 12972
scaler_y fitted rows  = 12972
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\mpnn_pretraining_internal_split_indices.npz


In [4]:
# Cell 3 | 定义 MPNN 编码器和黏度监督预训练模型


class MPNNEncoder(nn.Module):
    """将一个离子分子图编码为 64 维图表征。"""

    def __init__(
        self,
        node_dim=7,
        edge_dim=4,
        hidden_dim=64,
    ):
        super().__init__()

        self.lin0 = nn.Linear(
            node_dim,
            hidden_dim,
        )

        edge_network = nn.Sequential(
            nn.Linear(
                edge_dim,
                32,
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                hidden_dim * hidden_dim,
            ),
        )

        self.conv = NNConv(
            hidden_dim,
            hidden_dim,
            edge_network,
            aggr="mean",
        )

        self.lin1 = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

    def forward(self, data):
        node_state = self.lin0(
            data.x.float()
        )

        node_state = torch.relu(
            self.conv(
                node_state,
                data.edge_index,
                data.edge_attr.float(),
            )
        )

        node_state = self.lin1(
            node_state
        )

        graph_embedding = global_mean_pool(
            node_state,
            data.batch,
        )

        return graph_embedding


class ILPredictor(nn.Module):
    """双分支 MPNN 编码器及黏度监督预测头。"""

    def __init__(self):
        super().__init__()

        self.cat_encoder = MPNNEncoder(
            node_dim=7,
            edge_dim=4,
            hidden_dim=64,
        )

        self.ani_encoder = MPNNEncoder(
            node_dim=7,
            edge_dim=4,
            hidden_dim=64,
        )

        # 64维阳离子 + 64维阴离子 + 2维T/P
        self.mlp = nn.Sequential(
            nn.Linear(
                130,
                64,
            ),
            nn.ReLU(),
            nn.Linear(
                64,
                32,
            ),
            nn.ReLU(),
            nn.Linear(
                32,
                1,
            ),
        )

    def extract_features(
        self,
        cat_data,
        ani_data,
        env_data,
    ):
        """提取 130 维中间特征，不经过最终预测头。"""
        cat_embedding = self.cat_encoder(
            cat_data
        )

        ani_embedding = self.ani_encoder(
            ani_data
        )

        features = torch.cat(
            [
                cat_embedding,
                ani_embedding,
                env_data,
            ],
            dim=1,
        )

        return features

    def forward(
        self,
        cat_data,
        ani_data,
        env_data,
    ):
        features = self.extract_features(
            cat_data,
            ani_data,
            env_data,
        )

        prediction = self.mlp(
            features
        )

        return prediction


model = ILPredictor().to(
    device
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("模型架构定义完成 ✓")
print("cation embedding dim =", 64)
print("anion embedding dim  =", 64)
print("T/P feature dim      =", 2)
print("descriptor dim       =", 130)
print("total parameters     =", total_parameters)
print("trainable parameters =", trainable_parameters)

模型架构定义完成 ✓
cation embedding dim = 64
anion embedding dim  = 64
T/P feature dim      = 2
descriptor dim       = 130
total parameters     = 298817
trainable parameters = 298817


In [5]:
# Cell 4 | 训练 MPNN 并恢复验证集最优模型


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5,
)

criterion = nn.MSELoss()

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=1e-7,
    )
)

MAX_EPOCHS = 150
EARLY_STOP_PATIENCE = 20
MIN_DELTA = 1e-6

best_val_loss = float("inf")
best_epoch = 0
best_state_dict = None
no_improve_count = 0

training_history = []

print(
    "开始黏度 lg_eta 的 MPNN 特征预训练"
    f"（设备：{device}）"
)

print(
    f"MAX_EPOCHS={MAX_EPOCHS}, "
    f"EarlyStopping patience="
    f"{EARLY_STOP_PATIENCE}"
)

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):
    # 训练阶段
    model.train()

    total_train_loss = 0.0

    for (
        cat_data,
        ani_data,
        env_data,
        target,
    ) in train_loader:

        cat_data = cat_data.to(
            device
        )

        ani_data = ani_data.to(
            device
        )

        env_data = env_data.to(
            device
        )

        target = target.to(
            device
        )

        optimizer.zero_grad()

        prediction = model(
            cat_data,
            ani_data,
            env_data,
        )

        loss = criterion(
            prediction,
            target,
        )

        loss.backward()

        optimizer.step()

        total_train_loss += (
            loss.item()
            * target.shape[0]
        )

    average_train_loss = (
        total_train_loss
        / len(train_dataset)
    )

    # 验证阶段
    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():
        for (
            cat_data,
            ani_data,
            env_data,
            target,
        ) in val_loader:

            cat_data = cat_data.to(
                device
            )

            ani_data = ani_data.to(
                device
            )

            env_data = env_data.to(
                device
            )

            target = target.to(
                device
            )

            prediction = model(
                cat_data,
                ani_data,
                env_data,
            )

            val_loss = criterion(
                prediction,
                target,
            )

            total_val_loss += (
                val_loss.item()
                * target.shape[0]
            )

    average_val_loss = (
        total_val_loss
        / len(val_dataset)
    )

    scheduler.step(
        average_val_loss
    )

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    improved = (
        average_val_loss
        < best_val_loss - MIN_DELTA
    )

    if improved:
        best_val_loss = (
            average_val_loss
        )

        best_epoch = epoch

        best_state_dict = (
            copy.deepcopy(
                model.state_dict()
            )
        )

        no_improve_count = 0

    else:
        no_improve_count += 1

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": (
                average_train_loss
            ),
            "val_loss": (
                average_val_loss
            ),
            "best_val_loss": (
                best_val_loss
            ),
            "best_epoch": (
                best_epoch
            ),
            "learning_rate": (
                current_lr
            ),
            "improved": int(
                improved
            ),
            "no_improve_count": (
                no_improve_count
            ),
        }
    )

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch == MAX_EPOCHS
    ):
        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: "
            f"{average_train_loss:.6f} | "
            f"Val Loss: "
            f"{average_val_loss:.6f} | "
            f"Best Val: "
            f"{best_val_loss:.6f} "
            f"@ Epoch {best_epoch:03d} | "
            f"LR: {current_lr:.8f}"
        )

    if (
        no_improve_count
        >= EARLY_STOP_PATIENCE
    ):
        print(
            "Early stopping：连续 "
            f"{EARLY_STOP_PATIENCE} 轮 "
            "Val Loss 未明显下降，"
            f"在 Epoch {epoch:03d} 停止训练。"
        )

        break

epochs_ran = len(
    training_history
)

if best_state_dict is None:
    raise RuntimeError(
        "训练过程中没有保存到最优模型。"
    )

model.load_state_dict(
    best_state_dict
)

training_history_df = pd.DataFrame(
    training_history
)

training_history_df.to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("黏度 lg_eta 的 MPNN 预训练完成 ✓")
print("epochs actually ran  =", epochs_ran)
print("best epoch          =", best_epoch)
print("best val loss       =", best_val_loss)
print("saved:", TRAINING_HISTORY_PATH)

开始黏度 lg_eta 的 MPNN 特征预训练（设备：cpu）
MAX_EPOCHS=150, EarlyStopping patience=20
Epoch 001/150 | Train Loss: 0.769431 | Val Loss: 0.765801 | Best Val: 0.765801 @ Epoch 001 | LR: 0.00100000
Epoch 010/150 | Train Loss: 0.200944 | Val Loss: 0.202258 | Best Val: 0.202258 @ Epoch 010 | LR: 0.00100000
Epoch 020/150 | Train Loss: 0.125449 | Val Loss: 0.122708 | Best Val: 0.122708 @ Epoch 020 | LR: 0.00100000
Epoch 030/150 | Train Loss: 0.085050 | Val Loss: 0.090971 | Best Val: 0.090971 @ Epoch 030 | LR: 0.00100000
Epoch 040/150 | Train Loss: 0.072534 | Val Loss: 0.093942 | Best Val: 0.069726 @ Epoch 039 | LR: 0.00100000
Epoch 050/150 | Train Loss: 0.062724 | Val Loss: 0.070544 | Best Val: 0.064354 @ Epoch 049 | LR: 0.00100000
Epoch 060/150 | Train Loss: 0.045359 | Val Loss: 0.054400 | Best Val: 0.052206 @ Epoch 055 | LR: 0.00100000
Epoch 070/150 | Train Loss: 0.036132 | Val Loss: 0.047516 | Best Val: 0.044701 @ Epoch 066 | LR: 0.00100000
Epoch 080/150 | Train Loss: 0.038742 | Val Loss: 0.044744 | B

In [6]:
# Cell 5 | 计算内部训练集和验证集指标


def collect_predictions(
    data_loader,
):
    """收集标准化空间中的真实值和预测值。"""
    model.eval()

    true_scaled_list = []
    pred_scaled_list = []

    with torch.no_grad():
        for (
            cat_data,
            ani_data,
            env_data,
            target,
        ) in data_loader:

            cat_data = cat_data.to(
                device
            )

            ani_data = ani_data.to(
                device
            )

            env_data = env_data.to(
                device
            )

            prediction = model(
                cat_data,
                ani_data,
                env_data,
            )

            true_scaled_list.append(
                target.cpu().numpy()
            )

            pred_scaled_list.append(
                prediction.cpu().numpy()
            )

    true_scaled = np.vstack(
        true_scaled_list
    )

    pred_scaled = np.vstack(
        pred_scaled_list
    )

    true_original = (
        scaler_y
        .inverse_transform(
            true_scaled
        )
        .reshape(-1)
    )

    pred_original = (
        scaler_y
        .inverse_transform(
            pred_scaled
        )
        .reshape(-1)
    )

    return (
        true_original,
        pred_original,
    )


def calculate_regression_metrics(
    y_true,
    y_pred,
):
    """计算 lg_eta 空间中的回归指标。"""
    r2 = r2_score(
        y_true,
        y_pred,
    )

    rmse = float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    if (
        np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        pearson_r = np.nan

    else:
        pearson_r = float(
            np.corrcoef(
                y_true,
                y_pred,
            )[0, 1]
        )

    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "Pearson_r": pearson_r,
    }


train_true, train_pred = collect_predictions(
    train_eval_loader
)

val_true, val_pred = collect_predictions(
    val_loader
)

train_metrics = calculate_regression_metrics(
    train_true,
    train_pred,
)

val_metrics = calculate_regression_metrics(
    val_true,
    val_pred,
)

metrics_df = pd.DataFrame(
    [
        {
            "split": "internal_train",
            "target": "lg_eta",
            "n_samples": len(
                train_true
            ),
            **train_metrics,
        },
        {
            "split": "internal_validation",
            "target": "lg_eta",
            "n_samples": len(
                val_true
            ),
            **val_metrics,
        },
    ]
)

metrics_df.to_csv(
    PRETRAIN_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

val_rows = (
    model_rows.iloc[
        internal_val_idx
    ]
    .reset_index(drop=True)
    .copy()
)

max_val_target_alignment_error = float(
    np.max(
        np.abs(
            val_rows["lg_eta"]
            .to_numpy(dtype=float)
            - val_true
        )
    )
)

if (
    max_val_target_alignment_error
    > 1e-6
):
    raise RuntimeError(
        "内部验证集预测结果与原始行顺序不一致。"
    )

val_predictions_df = pd.DataFrame(
    {
        "feature_row_idx": (
            internal_val_idx
        ),
        "source_row_idx": (
            val_rows[
                "source_row_idx"
            ].to_numpy(dtype=int)
        ),
        "graph_list_idx": (
            val_rows[
                "graph_list_idx"
            ].to_numpy(dtype=int)
        ),
        "IL_key": (
            val_rows["IL_key"]
            .astype(str)
            .to_numpy()
        ),
        "cation_smiles": (
            val_rows[
                "cation_smiles"
            ]
            .astype(str)
            .to_numpy()
        ),
        "anion_smiles": (
            val_rows[
                "anion_smiles"
            ]
            .astype(str)
            .to_numpy()
        ),
        "T_rep": (
            val_rows["T_rep"]
            .to_numpy(dtype=float)
        ),
        "P_rep": (
            val_rows["P_rep"]
            .to_numpy(dtype=float)
        ),
        "lg_eta_true": val_true,
        "lg_eta_pred": val_pred,
        "residual_pred_minus_true": (
            val_pred - val_true
        ),
        "absolute_error": np.abs(
            val_pred - val_true
        ),
    }
)

val_predictions_df.to_csv(
    VAL_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("内部预训练指标计算完成 ✓")
print()
print(metrics_df.to_string(index=False))
print()
print(
    "max validation alignment error =",
    max_val_target_alignment_error,
)
print("saved:", PRETRAIN_METRICS_PATH)
print("saved:", VAL_PREDICTIONS_PATH)

内部预训练指标计算完成 ✓

              split target  n_samples       R2     RMSE      MAE  Pearson_r
     internal_train lg_eta      12972 0.988063 0.090280 0.055102   0.994041
internal_validation lg_eta       3244 0.980921 0.120626 0.066127   0.990441

max validation alignment error = 5.599553105639643e-07
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\mpnn_pretraining_metrics_eta.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\mpnn_pretraining_val_predictions_eta.csv


In [7]:
# Cell 6 | 提取并保存全部130维 MPNN特征

print(
    "开始提取基于 lg_eta 监督预训练得到的 "
    "130 维 MPNN 特征..."
)

model.eval()

feature_batches = []

with torch.no_grad():
    for (
        cat_data,
        ani_data,
        env_data,
        _,
    ) in full_loader:

        cat_data = cat_data.to(
            device
        )

        ani_data = ani_data.to(
            device
        )

        env_data = env_data.to(
            device
        )

        batch_features = (
            model.extract_features(
                cat_data,
                ani_data,
                env_data,
            )
        )

        feature_batches.append(
            batch_features
            .cpu()
            .numpy()
        )

final_features = np.vstack(
    feature_batches
).astype(np.float32)

expected_feature_shape = (
    len(model_rows),
    130,
)

if (
    final_features.shape
    != expected_feature_shape
):
    raise RuntimeError(
        "MPNN 特征矩阵维度错误："
        f"{final_features.shape}，"
        f"预期为 {expected_feature_shape}。"
    )

if not np.isfinite(
    final_features
).all():
    raise RuntimeError(
        "MPNN 特征矩阵中存在 NaN 或无穷值。"
    )

# 最后两维必须与标准化后的 T/P 完全对应
max_env_feature_difference = float(
    np.max(
        np.abs(
            final_features[
                :,
                128:130,
            ]
            - env_vars_scaled
        )
    )
)

if (
    max_env_feature_difference
    > 1e-6
):
    raise RuntimeError(
        "130维特征最后两维与标准化 T/P 不一致。"
    )

np.save(
    FEATURE_PATH,
    final_features,
)

feature_row_manifest = (
    model_rows[
        [
            "source_row_idx",
            "graph_list_idx",
            "IL_key",
            "cation_smiles",
            "anion_smiles",
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
        ]
    ]
    .copy()
)

feature_row_manifest.insert(
    0,
    "feature_row_idx",
    np.arange(
        len(feature_row_manifest),
        dtype=int,
    ),
)

feature_row_manifest.to_csv(
    FEATURE_ROW_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

np.savez_compressed(
    FEATURE_ALIGNMENT_PATH,

    feature_row_idx=np.arange(
        len(model_rows),
        dtype=int,
    ),

    source_row_idx=(
        model_rows[
            "source_row_idx"
        ]
        .to_numpy(dtype=int)
    ),

    graph_list_idx=(
        model_rows[
            "graph_list_idx"
        ]
        .to_numpy(dtype=int)
    ),

    internal_train_feature_row_idx=(
        internal_train_idx
    ),

    internal_val_feature_row_idx=(
        internal_val_idx
    ),
)

print("130维 MPNN 特征提取完成 ✓")
print("supervised target      =", "lg_eta")
print("feature matrix shape   =", final_features.shape)
print(
    "max T/P alignment diff =",
    max_env_feature_difference,
)
print("saved:", FEATURE_PATH)
print("saved:", FEATURE_ROW_MANIFEST_PATH)
print("saved:", FEATURE_ALIGNMENT_PATH)

开始提取基于 lg_eta 监督预训练得到的 130 维 MPNN 特征...
130维 MPNN 特征提取完成 ✓
supervised target      = lg_eta
feature matrix shape   = (16216, 130)
max T/P alignment diff = 0.0
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\mpnn_features_eta.npy
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\mpnn_features_eta_row_manifest.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\mpnn_features_eta_alignment.npz


In [8]:
# Cell 7 | 保存 MPNN 特征提取器、标准化器和 metadata


def calculate_sha256(
    path: Path,
) -> str:
    """计算文件 SHA256。"""
    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(
                block
            )

    return sha256.hexdigest()


state_dict_cpu = {
    key: value.detach().cpu()
    for key, value
    in model.state_dict().items()
}

checkpoint = {
    "property": "viscosity",
    "target_col": "lg_eta",
    "representation_method": "MPNN",

    "model_state_dict": (
        state_dict_cpu
    ),

    "model_class": (
        "ILPredictor"
    ),

    "encoder_class": (
        "MPNNEncoder"
    ),

    "node_dim": 7,
    "edge_dim": 4,
    "hidden_dim": 64,
    "cation_embedding_dim": 64,
    "anion_embedding_dim": 64,
    "environment_dim": 2,
    "descriptor_dim": 130,

    "random_seed": int(
        SEED
    ),

    "internal_train_rows": int(
        len(internal_train_idx)
    ),

    "internal_val_rows": int(
        len(internal_val_idx)
    ),

    "best_epoch": int(
        best_epoch
    ),

    "best_val_loss": float(
        best_val_loss
    ),

    "epochs_ran": int(
        epochs_ran
    ),
}

torch.save(
    checkpoint,
    MODEL_PATH,
)

joblib.dump(
    scaler_X,
    TP_SCALER_PATH,
)

joblib.dump(
    scaler_y,
    TARGET_SCALER_PATH,
)

metadata = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "project_root": str(
        ROOT
    ),

    "experiment_root": str(
        EXPERIMENT_ROOT
    ),

    "task": (
        "viscosity_10point_holdout_"
        "mpnn_feature_pretraining"
    ),

    "property": "viscosity",

    "target_col": "lg_eta",

    "representation_method": "MPNN",

    "input_files": {
        "cation_graphs": str(
            CATION_GRAPHS_PATH
        ),

        "anion_graphs": str(
            ANION_GRAPHS_PATH
        ),

        "graph_manifest": str(
            GRAPH_MANIFEST_PATH
        ),

        "df_ok": str(
            DF_OK_PATH
        ),

        "feature_representation_manifest": str(
            FEATURE_REPRESENTATION_MANIFEST_PATH
        ),
    },

    "input_hashes": {
        "graph_manifest_sha256": calculate_sha256(
            GRAPH_MANIFEST_PATH
        ),

        "df_ok_sha256": calculate_sha256(
            DF_OK_PATH
        ),
    },

    "n_total_rows": int(
        len(model_rows)
    ),

    "internal_split": {
        "method": (
            "random 80:20 split"
        ),

        "random_seed": int(
            SEED
        ),

        "train_rows": int(
            len(internal_train_idx)
        ),

        "validation_rows": int(
            len(internal_val_idx)
        ),

        "split_indices_file": str(
            INTERNAL_SPLIT_PATH
        ),
    },

    "external_holdout": {
        "n_points": 10,

        "used_in_pretraining": False,

        "note": (
            "The external 10-point holdout set "
            "was removed before this notebook."
        ),
    },

    "standardization": {
        "tp_scaler": (
            "StandardScaler fitted only on "
            "the internal training subset"
        ),

        "target_scaler": (
            "StandardScaler fitted only on "
            "the internal training subset"
        ),

        "tp_scaler_path": str(
            TP_SCALER_PATH
        ),

        "target_scaler_path": str(
            TARGET_SCALER_PATH
        ),
    },

    "architecture": {
        "node_feature_dim": 7,
        "edge_feature_dim": 4,
        "cation_embedding_dim": 64,
        "anion_embedding_dim": 64,
        "environment_dim": 2,
        "descriptor_dim": 130,

        "descriptor_composition": (
            "64-dimensional cation embedding + "
            "64-dimensional anion embedding + "
            "2-dimensional scaled T/P"
        ),
    },

    "training": {
        "optimizer": "Adam",
        "initial_learning_rate": 0.001,
        "weight_decay": 1e-5,
        "loss": "MSELoss",
        "scheduler": (
            "ReduceLROnPlateau"
        ),
        "scheduler_factor": 0.5,
        "scheduler_patience": 8,
        "maximum_epochs": int(
            MAX_EPOCHS
        ),
        "early_stopping_patience": int(
            EARLY_STOP_PATIENCE
        ),
        "minimum_delta": float(
            MIN_DELTA
        ),
        "epochs_ran": int(
            epochs_ran
        ),
        "best_epoch": int(
            best_epoch
        ),
        "best_val_loss_standardized": float(
            best_val_loss
        ),
        "history_file": str(
            TRAINING_HISTORY_PATH
        ),
        "metrics_file": str(
            PRETRAIN_METRICS_PATH
        ),
        "validation_predictions_file": str(
            VAL_PREDICTIONS_PATH
        ),
    },

    "outputs": {
        "feature_file": str(
            FEATURE_PATH
        ),

        "feature_file_sha256": calculate_sha256(
            FEATURE_PATH
        ),

        "feature_row_manifest": str(
            FEATURE_ROW_MANIFEST_PATH
        ),

        "feature_alignment": str(
            FEATURE_ALIGNMENT_PATH
        ),

        "model_path": str(
            MODEL_PATH
        ),

        "descriptor_dim": 130,
    },

    "internal_metrics": (
        metrics_df.to_dict(
            orient="records"
        )
    ),
}

with open(
    MODEL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("MPNN 特征提取器与标准化器保存完成 ✓")
print("property              =", "viscosity")
print("target                =", "lg_eta")
print("descriptor dimension  =", 130)
print("best epoch           =", best_epoch)
print("best val loss        =", best_val_loss)
print("model                =", MODEL_PATH)
print("T/P scaler           =", TP_SCALER_PATH)
print("target scaler        =", TARGET_SCALER_PATH)
print("metadata             =", MODEL_METADATA_PATH)

MPNN 特征提取器与标准化器保存完成 ✓
property              = viscosity
target                = lg_eta
descriptor dimension  = 130
best epoch           = 149
best val loss        = 0.02130936918626327
model                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\models\mpnn_feature_extractors\mpnn_feature_extractor_viscosity.pt
T/P scaler           = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\models\mpnn_feature_extractors\tp_scaler_viscosity.joblib
target scaler        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\models\mpnn_feature_extractors\target_scaler_viscosity.joblib
metadata             = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\models\mpnn_feature_extractors\mpnn_feature_extractor_viscosity_metadata.json


In [9]:
# Cell 8 | 最终检查全部输出和数据衔接

expected_output_files = [
    FEATURE_PATH,
    FEATURE_ROW_MANIFEST_PATH,
    FEATURE_ALIGNMENT_PATH,
    INTERNAL_SPLIT_PATH,
    TRAINING_HISTORY_PATH,
    PRETRAIN_METRICS_PATH,
    VAL_PREDICTIONS_PATH,
    MODEL_PATH,
    TP_SCALER_PATH,
    TARGET_SCALER_PATH,
    MODEL_METADATA_PATH,
]

missing_output_files = [
    path
    for path in expected_output_files
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "以下输出文件缺失：\n"
        + "\n".join(
            str(path)
            for path in missing_output_files
        )
    )

feature_check = np.load(
    FEATURE_PATH
)

if feature_check.shape != (
    len(model_rows),
    130,
):
    raise RuntimeError(
        "mpnn_features_eta.npy 维度错误："
        f"{feature_check.shape}"
    )

if not np.isfinite(
    feature_check
).all():
    raise RuntimeError(
        "mpnn_features_eta.npy 中存在 "
        "NaN 或无穷值。"
    )

feature_row_manifest_check = pd.read_csv(
    FEATURE_ROW_MANIFEST_PATH,
    encoding="utf-8-sig",
)

if len(
    feature_row_manifest_check
) != len(model_rows):
    raise RuntimeError(
        "特征行清单与特征矩阵行数不一致。"
    )

expected_feature_row_idx = np.arange(
    len(model_rows),
    dtype=int,
)

actual_feature_row_idx = (
    feature_row_manifest_check[
        "feature_row_idx"
    ]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    actual_feature_row_idx,
    expected_feature_row_idx,
):
    raise RuntimeError(
        "feature_row_idx 不连续或顺序错误。"
    )

if not np.array_equal(
    feature_row_manifest_check[
        "source_row_idx"
    ].to_numpy(dtype=int),
    model_rows[
        "source_row_idx"
    ].to_numpy(dtype=int),
):
    raise RuntimeError(
        "特征矩阵与 source_row_idx 对齐错误。"
    )

with np.load(
    FEATURE_ALIGNMENT_PATH
) as alignment_check:

    alignment_source_idx = (
        alignment_check[
            "source_row_idx"
        ]
        .astype(int)
    )

    alignment_graph_idx = (
        alignment_check[
            "graph_list_idx"
        ]
        .astype(int)
    )

if not np.array_equal(
    alignment_source_idx,
    model_rows[
        "source_row_idx"
    ].to_numpy(dtype=int),
):
    raise RuntimeError(
        "特征 alignment 文件中的 "
        "source_row_idx 不一致。"
    )

if not np.array_equal(
    alignment_graph_idx,
    model_rows[
        "graph_list_idx"
    ].to_numpy(dtype=int),
):
    raise RuntimeError(
        "特征 alignment 文件中的 "
        "graph_list_idx 不一致。"
    )

max_environment_difference_check = float(
    np.max(
        np.abs(
            feature_check[
                :,
                128:130,
            ]
            - env_vars_scaled
        )
    )
)

if (
    max_environment_difference_check
    > 1e-6
):
    raise RuntimeError(
        "特征矩阵最后两维与标准化 T/P 不一致。"
    )

history_check = pd.read_csv(
    TRAINING_HISTORY_PATH,
    encoding="utf-8-sig",
)

metrics_check = pd.read_csv(
    PRETRAIN_METRICS_PATH,
    encoding="utf-8-sig",
)

val_predictions_check = pd.read_csv(
    VAL_PREDICTIONS_PATH,
    encoding="utf-8-sig",
)

if len(history_check) != epochs_ran:
    raise RuntimeError(
        "训练历史行数与实际训练轮数不一致。"
    )

history_best_epoch = int(
    history_check.loc[
        history_check[
            "val_loss"
        ].idxmin(),
        "epoch",
    ]
)

history_best_val_loss = float(
    history_check[
        "val_loss"
    ].min()
)

if history_best_epoch != best_epoch:
    raise RuntimeError(
        "训练历史中的最佳轮次 "
        "与内存记录不一致。"
    )

if not np.isclose(
    history_best_val_loss,
    best_val_loss,
    rtol=1e-7,
    atol=1e-9,
):
    raise RuntimeError(
        "训练历史中的最佳验证损失 "
        "与内存记录不一致。"
    )

if len(
    val_predictions_check
) != len(internal_val_idx):
    raise RuntimeError(
        "内部验证集预测文件行数错误。"
    )

required_metric_splits = {
    "internal_train",
    "internal_validation",
}

actual_metric_splits = set(
    metrics_check[
        "split"
    ].astype(str)
)

if (
    actual_metric_splits
    != required_metric_splits
):
    raise RuntimeError(
        "预训练指标文件中的数据集名称错误。"
    )

checkpoint_check = load_torch_object(
    MODEL_PATH
)

required_checkpoint_keys = [
    "model_state_dict",
    "property",
    "target_col",
    "descriptor_dim",
    "best_epoch",
    "best_val_loss",
]

missing_checkpoint_keys = [
    key
    for key in required_checkpoint_keys
    if key not in checkpoint_check
]

if missing_checkpoint_keys:
    raise RuntimeError(
        "模型 checkpoint 缺少字段："
        f"{missing_checkpoint_keys}"
    )

if (
    checkpoint_check["property"]
    != "viscosity"
):
    raise RuntimeError(
        "checkpoint 中的 property "
        "不是 viscosity。"
    )

if (
    checkpoint_check["target_col"]
    != "lg_eta"
):
    raise RuntimeError(
        "checkpoint 中的 target_col "
        "不是 lg_eta。"
    )

if int(
    checkpoint_check[
        "descriptor_dim"
    ]
) != 130:
    raise RuntimeError(
        "checkpoint 中的 descriptor_dim "
        "不是 130。"
    )

model_structure_check = (
    ILPredictor()
)

model_structure_check.load_state_dict(
    checkpoint_check[
        "model_state_dict"
    ],
    strict=True,
)

scaler_X_check = joblib.load(
    TP_SCALER_PATH
)

scaler_y_check = joblib.load(
    TARGET_SCALER_PATH
)

if int(
    scaler_X_check.n_features_in_
) != 2:
    raise RuntimeError(
        "T/P 标准化器的输入维度不是 2。"
    )

if int(
    scaler_y_check.n_features_in_
) != 1:
    raise RuntimeError(
        "目标值标准化器的输入维度不是 1。"
    )

with open(
    MODEL_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    metadata_check = json.load(file)

if (
    metadata_check.get("property")
    != "viscosity"
):
    raise RuntimeError(
        "metadata 中的 property "
        "不是 viscosity。"
    )

if (
    metadata_check.get("target_col")
    != "lg_eta"
):
    raise RuntimeError(
        "metadata 中的 target_col "
        "不是 lg_eta。"
    )

if (
    metadata_check.get(
        "representation_method"
    )
    != "MPNN"
):
    raise RuntimeError(
        "metadata 中的 representation_method "
        "不是 MPNN。"
    )

if (
    metadata_check[
        "external_holdout"
    ][
        "used_in_pretraining"
    ]
    is not False
):
    raise RuntimeError(
        "metadata 错误地标记为使用了 "
        "外部10点留出数据。"
    )

train_metric_row = (
    metrics_check[
        metrics_check["split"].eq(
            "internal_train"
        )
    ]
    .iloc[0]
)

val_metric_row = (
    metrics_check[
        metrics_check["split"].eq(
            "internal_validation"
        )
    ]
    .iloc[0]
)

print("=" * 84)
print("全部 MPNN 特征预训练输出检查通过 ✓")
print("=" * 84)

print("\n输入数据：")
print("model rows             =", len(model_rows))
print("unique ILs             =", model_rows["IL_key"].nunique())
print("unique cations         =", model_rows["cation_smiles"].nunique())
print("unique anions          =", model_rows["anion_smiles"].nunique())

print("\n内部预训练划分：")
print("internal train rows    =", len(internal_train_idx))
print("internal val rows      =", len(internal_val_idx))
print("external holdout used  =", False)

print("\n训练结果：")
print("epochs ran             =", epochs_ran)
print("best epoch             =", best_epoch)
print("best standardized loss =", best_val_loss)

print("\n内部训练集指标（lg_eta）：")
print("R2                      =", float(train_metric_row["R2"]))
print("RMSE                    =", float(train_metric_row["RMSE"]))
print("MAE                     =", float(train_metric_row["MAE"]))
print("Pearson r               =", float(train_metric_row["Pearson_r"]))

print("\n内部验证集指标（lg_eta）：")
print("R2                      =", float(val_metric_row["R2"]))
print("RMSE                    =", float(val_metric_row["RMSE"]))
print("MAE                     =", float(val_metric_row["MAE"]))
print("Pearson r               =", float(val_metric_row["Pearson_r"]))

print("\n特征输出：")
print("representation          =", "MPNN")
print("feature matrix shape    =", feature_check.shape)
print("feature dtype           =", feature_check.dtype)
print(
    "max T/P alignment diff  =",
    max_environment_difference_check,
)

print("\n输出文件完整路径：")
for path in expected_output_files:
    print(path)

全部 MPNN 特征预训练输出检查通过 ✓

输入数据：
model rows             = 16216
unique ILs             = 1295
unique cations         = 583
unique anions          = 202

内部预训练划分：
internal train rows    = 12972
internal val rows      = 3244
external holdout used  = False

训练结果：
epochs ran             = 150
best epoch             = 149
best standardized loss = 0.02130936918626327

内部训练集指标（lg_eta）：
R2                      = 0.9880634546279908
RMSE                    = 0.0902803634106184
MAE                     = 0.055101852864027
Pearson r               = 0.99404121823388

内部验证集指标（lg_eta）：
R2                      = 0.9809208512306212
RMSE                    = 0.1206256061993324
MAE                     = 0.0661265477538108
Pearson r               = 0.9904407089822262

特征输出：
representation          = MPNN
feature matrix shape    = (16216, 130)
feature dtype           = float32
max T/P alignment diff  = 0.0

输出文件完整路径：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_comp

In [10]:
# Cell 9 | 筛选内部验证集中的大误差样本

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start_path=".") -> Path:
    """向上查找项目根目录。"""
    current_path = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。"
    )


ROOT = find_project_root(".")

EXPERIMENT_ROOT = (
    ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

PRETRAIN_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "mpnn_pretraining"
)

ERROR_REVIEW_DIR = (
    PRETRAIN_DIR
    / "error_review"
)

ERROR_REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VAL_PREDICTIONS_PATH = (
    PRETRAIN_DIR
    / "mpnn_pretraining_val_predictions_eta.csv"
)

if not VAL_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"找不到内部验证集预测文件：\n"
        f"{VAL_PREDICTIONS_PATH}"
    )

df_val = pd.read_csv(
    VAL_PREDICTIONS_PATH,
    encoding="utf-8-sig",
)

required_cols = [
    "feature_row_idx",
    "source_row_idx",
    "graph_list_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "lg_eta_true",
    "lg_eta_pred",
    "residual_pred_minus_true",
    "absolute_error",
]

missing_cols = [
    col
    for col in required_cols
    if col not in df_val.columns
]

if missing_cols:
    raise ValueError(
        f"验证集预测文件缺少字段：{missing_cols}"
    )

numeric_cols = [
    "feature_row_idx",
    "source_row_idx",
    "graph_list_idx",
    "T_rep",
    "P_rep",
    "lg_eta_true",
    "lg_eta_pred",
    "residual_pred_minus_true",
    "absolute_error",
]

for col in numeric_cols:
    df_val[col] = pd.to_numeric(
        df_val[col],
        errors="coerce",
    )

if df_val[numeric_cols].isna().any().any():
    raise ValueError(
        "验证集预测文件中的数值字段存在缺失。"
    )

# 将 lg_eta 转回 eta，便于人工核查
df_val["eta_true_mPa_s"] = np.power(
    10.0,
    df_val["lg_eta_true"],
)

df_val["eta_pred_mPa_s"] = np.power(
    10.0,
    df_val["lg_eta_pred"],
)

# 绝对误差对应的黏度倍数差
df_val["error_factor"] = np.power(
    10.0,
    df_val["absolute_error"],
)

df_val["error_direction"] = np.where(
    df_val["residual_pred_minus_true"] > 0,
    "预测偏高",
    np.where(
        df_val["residual_pred_minus_true"] < 0,
        "预测偏低",
        "无偏差",
    ),
)

# source_row_idx 从0开始；
# 在 Excel 中打开 ml_ready_A.csv 时，需要加2得到实际行号
df_val["ml_ready_excel_row"] = (
    df_val["source_row_idx"]
    .astype(int)
    + 2
)

df_val["review_level"] = np.select(
    [
        df_val["absolute_error"] >= 0.50,
        df_val["absolute_error"] >= 0.30,
    ],
    [
        "优先核查",
        "关注",
    ],
    default="一般",
)

df_priority = (
    df_val[
        df_val["absolute_error"] >= 0.50
    ]
    .sort_values(
        "absolute_error",
        ascending=False,
    )
    .reset_index(drop=True)
)

df_watchlist = (
    df_val[
        df_val["absolute_error"] >= 0.30
    ]
    .sort_values(
        "absolute_error",
        ascending=False,
    )
    .reset_index(drop=True)
)

PRIORITY_REVIEW_PATH = (
    ERROR_REVIEW_DIR
    / "large_error_priority_review_ge_0p50.csv"
)

WATCHLIST_PATH = (
    ERROR_REVIEW_DIR
    / "large_error_watchlist_ge_0p30.csv"
)

df_priority.to_csv(
    PRIORITY_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

df_watchlist.to_csv(
    WATCHLIST_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("大误差样本筛选完成 ✓")
print("validation rows       =", len(df_val))
print(
    "absolute error ≥ 0.30 =",
    len(df_watchlist),
)
print(
    "absolute error ≥ 0.50 =",
    len(df_priority),
)
print(
    "maximum absolute error =",
    df_val["absolute_error"].max(),
)
print("saved:", PRIORITY_REVIEW_PATH)
print("saved:", WATCHLIST_PATH)

display(
    df_priority[
        [
            "source_row_idx",
            "ml_ready_excel_row",
            "IL_key",
            "T_rep",
            "P_rep",
            "lg_eta_true",
            "lg_eta_pred",
            "absolute_error",
            "error_factor",
            "error_direction",
        ]
    ].head(20)
)

大误差样本筛选完成 ✓
validation rows       = 3244
absolute error ≥ 0.30 = 80
absolute error ≥ 0.50 = 33
maximum absolute error = 1.7040964
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\error_review\large_error_priority_review_ge_0p50.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\error_review\large_error_watchlist_ge_0p30.csv


,source_row_idx,ml_ready_excel_row,IL_key,T_rep,P_rep,lg_eta_true,lg_eta_pred,absolute_error,error_factor,error_direction
0,5652,5654,CCCCOn1cc[n+](CCCC)c1||Cc1ccc(S(=O)(=O)[O-])cc1,298.150,1.01325,1.863323,3.567419,1.704096,50.593695,预测偏高
1,11025,11027,CCOC(=O)Cn1cc[n+](C)c1||[Cl-],298.150,1.00000,2.542826,3.974647,1.431821,27.028447,预测偏高
2,10149,10151,CCCCn1cc[n+](CCCC)c1||CS(=O)(=O)[O-],298.150,1.01325,1.503791,2.827077,1.323286,21.051658,预测偏高
3,11635,11637,CC[NH+](CC)CC||O=P([O-])(O)O,303.150,1.01000,1.756636,3.003563,1.246927,17.657402,预测偏高
4,15253,15255,C[n+]1ccn(CCCN)c1||O=C([O-])c1ncc[nH]1,313.000,1.01325,1.736397,2.953880,1.217483,16.499976,预测偏高
5,5040,5042,CCCCC[N+](C)(C)CCC[N+](C)(C)CCOCC||O=S(=O)([N-...,303.100,1.01325,3.694518,2.745664,0.948854,8.889012,预测偏低
6,13975,13977,COC(=O)[C@H]([NH3+])Cc1ccccc1||O=S(=O)([N-]S(=...,298.100,1.01325,3.232996,4.125982,0.892986,7.816027,预测偏高
7,11201,11203,CC[N+](C)(C)CCO||CCOS(=O)(=O)[O-],298.150,1.01325,3.208710,2.354677,0.854033,7.145514,预测偏低
8,8,10,C1CC[NH2+]C1||O=[N+]([O-])[O-],298.150,1.01325,0.716003,1.518259,0.802256,6.342433,预测偏高
9,5042,5044,CCCCC[N+](C)(C)CCC[N+](C)(C)CCOCC||O=S(=O)([N-...,313.100,1.01325,3.323046,2.534025,0.789021,6.152070,预测偏低


In [11]:
# Cell 10 | 整理大误差离子液体的完整温度和压力序列

GRAPH_MANIFEST_PATH = (
    EXPERIMENT_ROOT
    / "features"
    / "mpnn_graph"
    / "graph_manifest.csv"
)

if not GRAPH_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"找不到图结构清单：\n"
        f"{GRAPH_MANIFEST_PATH}"
    )

df_graph = pd.read_csv(
    GRAPH_MANIFEST_PATH,
    encoding="utf-8-sig",
)

required_graph_cols = [
    "source_row_idx",
    "graph_list_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

missing_graph_cols = [
    col
    for col in required_graph_cols
    if col not in df_graph.columns
]

if missing_graph_cols:
    raise ValueError(
        f"graph_manifest.csv 缺少字段："
        f"{missing_graph_cols}"
    )

for col in [
    "source_row_idx",
    "graph_list_idx",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    df_graph[col] = pd.to_numeric(
        df_graph[col],
        errors="coerce",
    )

priority_il_keys = (
    df_priority["IL_key"]
    .drop_duplicates()
    .tolist()
)

# 取出这些离子液体在完整数据中的全部实验点
df_large_error_series = (
    df_graph[
        df_graph["IL_key"].isin(
            priority_il_keys
        )
    ]
    .copy()
)

prediction_cols = [
    "source_row_idx",
    "lg_eta_true",
    "lg_eta_pred",
    "residual_pred_minus_true",
    "absolute_error",
    "error_factor",
    "error_direction",
    "review_level",
]

df_large_error_series = (
    df_large_error_series
    .merge(
        df_val[prediction_cols],
        on="source_row_idx",
        how="left",
        validate="one_to_one",
    )
)

df_large_error_series[
    "in_internal_validation"
] = (
    df_large_error_series[
        "lg_eta_true"
    ].notna()
)

df_large_error_series[
    "is_priority_error"
] = (
    df_large_error_series[
        "absolute_error"
    ].ge(0.50)
)

df_large_error_series[
    "is_watchlist_error"
] = (
    df_large_error_series[
        "absolute_error"
    ].ge(0.30)
)

df_large_error_series[
    "ml_ready_excel_row"
] = (
    df_large_error_series[
        "source_row_idx"
    ].astype(int)
    + 2
)

df_large_error_series = (
    df_large_error_series
    .sort_values(
        [
            "IL_key",
            "P_rep",
            "T_rep",
        ]
    )
    .reset_index(drop=True)
)

IL_SERIES_PATH = (
    ERROR_REVIEW_DIR
    / "large_error_il_full_series.csv"
)

df_large_error_series.to_csv(
    IL_SERIES_PATH,
    index=False,
    encoding="utf-8-sig",
)

# 汇总每一种离子液体的情况
df_il_summary = (
    df_large_error_series
    .groupby(
        "IL_key",
        as_index=False,
    )
    .agg(
        cation_smiles=(
            "cation_smiles",
            "first",
        ),
        anion_smiles=(
            "anion_smiles",
            "first",
        ),
        n_total_experimental_points=(
            "source_row_idx",
            "size",
        ),
        n_internal_validation_points=(
            "in_internal_validation",
            "sum",
        ),
        n_priority_error_points=(
            "is_priority_error",
            "sum",
        ),
        n_watchlist_error_points=(
            "is_watchlist_error",
            "sum",
        ),
        T_min_K=(
            "T_rep",
            "min",
        ),
        T_max_K=(
            "T_rep",
            "max",
        ),
        P_min_bar=(
            "P_rep",
            "min",
        ),
        P_max_bar=(
            "P_rep",
            "max",
        ),
        eta_min_mPa_s=(
            "eta",
            "min",
        ),
        eta_max_mPa_s=(
            "eta",
            "max",
        ),
        maximum_absolute_error=(
            "absolute_error",
            "max",
        ),
    )
)

df_il_summary = (
    df_il_summary
    .sort_values(
        [
            "maximum_absolute_error",
            "n_priority_error_points",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

IL_SUMMARY_PATH = (
    ERROR_REVIEW_DIR
    / "large_error_il_summary.csv"
)

df_il_summary.to_csv(
    IL_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("大误差离子液体序列整理完成 ✓")
print(
    "priority samples      =",
    len(df_priority),
)
print(
    "affected unique ILs   =",
    len(priority_il_keys),
)
print(
    "full-series rows      =",
    len(df_large_error_series),
)
print("saved:", IL_SERIES_PATH)
print("saved:", IL_SUMMARY_PATH)

display(
    df_il_summary.head(20)
)

大误差离子液体序列整理完成 ✓
priority samples      = 33
affected unique ILs   = 29
full-series rows      = 224
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\error_review\large_error_il_full_series.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\error_review\large_error_il_summary.csv


,IL_key,cation_smiles,anion_smiles,n_total_experimental_points,n_internal_validation_points,n_priority_error_points,n_watchlist_error_points,T_min_K,T_max_K,P_min_bar,P_max_bar,eta_min_mPa_s,eta_max_mPa_s,maximum_absolute_error
0,CCCCOn1cc[n+](CCCC)c1||Cc1ccc(S(=O)(=O)[O-])cc1,CCCCOn1cc[n+](CCCC)c1,Cc1ccc(S(=O)(=O)[O-])cc1,1,1,1,1,298.15,298.15,1.01325,1.01325,73.00,7.300000e+01,1.704096
1,CCOC(=O)Cn1cc[n+](C)c1||[Cl-],CCOC(=O)Cn1cc[n+](C)c1,[Cl-],1,1,1,1,298.15,298.15,1.00000,1.00000,349.00,3.490000e+02,1.431821
2,CCCCn1cc[n+](CCCC)c1||CS(=O)(=O)[O-],CCCCn1cc[n+](CCCC)c1,CS(=O)(=O)[O-],1,1,1,1,298.15,298.15,1.01325,1.01325,31.90,3.190000e+01,1.323286
3,CC[NH+](CC)CC||O=P([O-])(O)O,CC[NH+](CC)CC,O=P([O-])(O)O,9,1,1,1,298.15,358.00,1.01000,1.01325,38.30,2.800000e+04,1.246927
4,C[n+]1ccn(CCCN)c1||O=C([O-])c1ncc[nH]1,C[n+]1ccn(CCCN)c1,O=C([O-])c1ncc[nH]1,1,1,1,1,313.00,313.00,1.01325,1.01325,54.50,5.450000e+01,1.217483
5,CCCCC[N+](C)(C)CCC[N+](C)(C)CCOCC||O=S(=O)([N-...,CCCCC[N+](C)(C)CCC[N+](C)(C)CCOCC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,7,2,2,2,303.10,333.10,1.01325,1.01325,531.00,4.949000e+03,0.948854
6,COC(=O)[C@H]([NH3+])Cc1ccccc1||O=S(=O)([N-]S(=...,COC(=O)[C@H]([NH3+])Cc1ccccc1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1,1,1,1,298.10,298.10,1.01325,1.01325,1710.00,1.710000e+03,0.892986
7,CC[N+](C)(C)CCO||CCOS(=O)(=O)[O-],CC[N+](C)(C)CCO,CCOS(=O)(=O)[O-],1,1,1,1,298.15,298.15,1.01325,1.01325,1617.00,1.617000e+03,0.854033
8,C1CC[NH2+]C1||O=[N+]([O-])[O-],C1CC[NH2+]C1,O=[N+]([O-])[O-],9,2,1,1,283.15,353.15,1.00000,1.01325,5.20,5.900000e+01,0.802256
9,CCCOC[N+](C)(C)CCO||O=S(=O)([N-]S(=O)(=O)C(F)(...,CCCOC[N+](C)(C)CCO,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1,1,1,1,298.10,298.10,1.01325,1.01325,84.00,8.400000e+01,0.725795


In [12]:
# Cell 11 | 生成大误差样本手工核查表

manual_review = (
    df_priority
    .copy()
)

manual_review.insert(
    0,
    "review_status",
    "待核查",
)

manual_review[
    "original_source_found"
] = ""

manual_review[
    "eta_value_checked"
] = ""

manual_review[
    "eta_unit_checked"
] = ""

manual_review[
    "temperature_checked"
] = ""

manual_review[
    "pressure_checked"
] = ""

manual_review[
    "cation_anion_checked"
] = ""

manual_review[
    "smiles_checked"
] = ""

manual_review[
    "duplicate_or_conflict_checked"
] = ""

manual_review[
    "same_IL_series_checked"
] = ""

manual_review[
    "decision"
] = ""

manual_review[
    "evidence_and_source_location"
] = ""

manual_review[
    "corrected_eta_mPa_s"
] = ""

manual_review[
    "corrected_T_K"
] = ""

manual_review[
    "corrected_P_bar"
] = ""

manual_review[
    "corrected_cation_smiles"
] = ""

manual_review[
    "corrected_anion_smiles"
] = ""

manual_review[
    "final_action"
] = ""

manual_review[
    "review_note"
] = ""

MANUAL_REVIEW_PATH = (
    ERROR_REVIEW_DIR
    / "large_error_manual_review_template.csv"
)

manual_review.to_csv(
    MANUAL_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("手工核查表生成完成 ✓")
print("saved:", MANUAL_REVIEW_PATH)

print("\n核查时 decision 只填写以下四种之一：")
print("1. keep_verified       原始数据核对无误，保留")
print("2. correct             确认存在错误，需要修正")
print("3. remove              确认不属于本数据集或无法恢复")
print("4. unresolved_keep     暂时无法确定，先保留")

print("\nfinal_action 只填写：")
print("keep / correct / remove")

手工核查表生成完成 ✓
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\mpnn_pretraining\error_review\large_error_manual_review_template.csv

核查时 decision 只填写以下四种之一：
1. keep_verified       原始数据核对无误，保留
2. correct             确认存在错误，需要修正
3. remove              确认不属于本数据集或无法恢复
4. unresolved_keep     暂时无法确定，先保留

final_action 只填写：
keep / correct / remove
